# Kalman filter

### Import libraries



In [31]:
#import libraries
import pandas as pd
import numpy as np
pd.set_option('display.float_format', lambda x: '%.3f' % x)
import matplotlib.pyplot as plt
import math
import os
import sys
import matplotlib.pyplot as plt
from sklearn.metrics import root_mean_squared_error





### Set up

In [32]:
class KalmanFilter(object):
    def __init__(self, F = None, B = None, H = None, Q = None, P = None, x0 = None):

        if(F is None or H is None):
            raise ValueError("Set proper system dynamics.")

        self.n = F.shape[1]
        self.m = H.shape[1]

        self.F = F
        self.H = H
        self.B = 0 if B is None else B
        self.Q = np.eye(self.n) if Q is None else Q
        self.P = np.eye(self.n) if P is None else P
        self.x = np.zeros((self.n, 1)) if x0 is None else x0

    def predict(self, u = 0):
        self.x = np.dot(self.F, self.x) + np.dot(self.B, u)
        self.P = np.dot(np.dot(self.F, self.P), self.F.T) + self.Q
        return self.x,self.P

    #def update(self, z,R):
    def update(self, z,R):
        y = z - np.dot(self.H, self.x)
        #S = self.R + np.dot(self.H, np.dot(self.P, self.H.T))
        S = R + np.dot(self.H, np.dot(self.P, self.H.T))
        K = np.dot(np.dot(self.P, self.H.T), np.linalg.inv(S))
        self.x = self.x + np.dot(K, y)
        I = np.eye(self.n)
        self.P = np.dot(np.dot(I - np.dot(K, self.H), self.P), 
        	#(I - np.dot(K, self.H)).T) + np.dot(np.dot(K, self.R), K.T)
            (I - np.dot(K, self.H)).T) + np.dot(np.dot(K, R), K.T)


In [33]:
#Set up options

#set folder name where the dataset is located
folder_data= 'imputed_data'
exp_test_complete=[4,13,5,12,0,10,3,11,2,9,6,7,1,8]
num_fold=7
fold1=[4,13]
fold2=[5,12]
fold3=[0,10]
fold4=[3,11]
fold5=[2,9]
fold6=[6,7]
fold7=[1,8]

list_fold=[fold1,fold2,fold3,fold4,fold5,fold6,fold7]

filter_list=['original','rain_1','rain_2','fog_1']

#SAMPLING TIME in seconds
dt =0.01
#to set the the starting point x0. if x0 is calculated as mean value of the initial state of the ground truth then set label_x0_mean=True.
label_x0_mean=True


### Kalman Filter elaboration
Training with the 'original' filter.

In [34]:
#set folder name where to save the elaborated data
folder_results= "Kalman_filter_results"


#to intialize the Results dataframe
exp = ['fold1','fold2','fold3','fold4','fold5','fold6','fold7']
final = [f"{f}_{e}" for f in filter_list for e in exp]
RMSE_results=pd.DataFrame(columns=['Kalman Filter_fold0','Kalman Filter_fold1','Kalman Filter_fold_combined'], index=final)

for fold in list_fold:   
    # Ground truth column
    gt_col = 'distance_tracker'
    # List of sensor columns 
    sensor_columns = ['distance_wheelchair_FT', 'distance_wheelchair_v5', 
                    'distance_drone_FT', 'distance_drone_v5', 'distance_range']
   
    
    # ==== Model training ====
    #Training part, Bias, Variance and initial mean value are calculated from the dataset of 'original filter'
    
    exp_test=[x for x in exp_test_complete if x not in fold]

    if label_x0_mean==True:
        initial_value_GT=[]
    for exp_nr in exp_test:
        folder_imputation = os.path.join(folder_data)

        data= pd.read_csv(f'{folder_imputation}/imputed_original_run_{exp_nr}.csv')
        data.rename(columns={'Unnamed: 0': 'sample'}, inplace=True)
        df=data[['distance_tracker','distance_wheelchair_FT','distance_wheelchair_v5','distance_drone_FT','distance_drone_v5','distance_range' ]].copy()
        if label_x0_mean==True:
            initial_value_GT.append(df.loc[0,'distance_tracker'])

        #to concatenate the data of several experiments
        if exp_nr==exp_test[0]:
            df_complete=df
        else:
            
            df.index=df.index+last_value
            df_complete=pd.concat([df_complete,df],axis=0)

        last_value=df.index[-1]
    
    df_training=df_complete.copy(deep=True)

    df_training[df_training > 5] = np.nan
        
    bias_variance = {}

    # Loop through each sensor column
    for sensor in sensor_columns:
        
        errors = df_training[sensor] - df_training[gt_col]
        bias = errors.mean()
        variance = errors.var()
        bias_variance[sensor] = {'bias': bias, 'variance': variance}

    bias_variance_df = pd.DataFrame(bias_variance).T
    if label_x0_mean==True:
        set_x0=np.mean(initial_value_GT)
    
    # ==== Model test ====
    F = np.array([[1, dt], [0, 1]])
    H = np.array([1, 0]).reshape(1, 2)
    #Q process noise uncertainty; in our case is the noise relate to X distance of the obstacle
    Var_a=1
    Q = np.array([[0, 0], [0, 0.01]])* Var_a
    #R measurment uncertainty; uncertainty related to the sensors noise

    R_range = np.array([bias_variance_df['variance']['distance_range']]).reshape(1, 1)
    R_w_FT = np.array([bias_variance_df['variance']['distance_wheelchair_FT']]).reshape(1, 1)
    R_w_v5 = np.array([bias_variance_df['variance']['distance_wheelchair_v5']]).reshape(1, 1)
    R_d_FT = np.array([bias_variance_df['variance']['distance_drone_FT']]).reshape(1, 1)
    R_d_v5 = np.array([bias_variance_df['variance']['distance_drone_v5']]).reshape(1, 1)



    for filter in filter_list:
        for exp_nr in fold:
            x0= np.array([set_x0,0]).reshape(2, 1)
            folder_imputation = os.path.join(folder_data)
            data= pd.read_csv(f'{folder_imputation}/imputed_{filter}_run_{exp_nr}.csv')
            data.rename(columns={'Unnamed: 0': 'sample'}, inplace=True)
            
            #ground thruth
            x =data['distance_tracker']

            #measures from the sensors (case 1 sensor, range sensors)
            measur_range = data['distance_range']-bias_variance_df['bias']['distance_range']
            measur_w_FT = data['distance_wheelchair_FT']-bias_variance_df['bias']['distance_wheelchair_FT']
            measur_w_v5 = data['distance_wheelchair_v5']-bias_variance_df['bias']['distance_wheelchair_v5']
            measur_d_FT = data['distance_drone_FT']-bias_variance_df['bias']['distance_drone_FT']
            measur_d_v5 = data['distance_drone_v5']-bias_variance_df['bias']['distance_drone_v5']

            #missingness of the sensors (case 1 sensor, range sensors)
            missing_range= data['missing_range']
            missing_w_FT= data['missing_w_FT']
            missing_w_v5= data['missing_w_v5']
            missing_d_FT= data['missing_d_FT']
            missing_d_v5= data['missing_d_v5']

            #KF initialization
            kf = KalmanFilter(F = F, H = H, Q = Q ,x0=x0)
            predictions = []
            uncertainty= []

            #KF elaboration
            for z in data.index:
                
                if missing_range[z]==0:
                    kf.update(measur_range[z],R_range)
                if missing_w_FT[z]==0:
                    kf.update(measur_w_FT[z],R_w_FT)
                if missing_w_v5[z]==0:
                    kf.update(measur_w_v5[z],R_w_v5)
                if missing_d_FT[z]==0:
                    kf.update(measur_d_FT[z],R_d_FT)
                if missing_d_v5[z]==0:
                    kf.update(measur_d_v5[z],R_d_v5)
                predictions.append(np.dot(H,  kf.predict()[0])[0][0])
                uncertainty.append(math.sqrt(kf.predict()[1][0][0]))

            # ==== SAVE RESULTS ====
            df_results=pd.DataFrame({'predictions':predictions,'uncertainty':uncertainty})
            newpath = f'{folder_results}' 
            if not os.path.exists(newpath):
                os.makedirs(newpath)
                        
            df_results.to_csv(f'{folder_results}/KF_results_{filter}_run_{exp_nr}.csv')



